In [2]:
using Pkg
Pkg.activate(expanduser("~/M466"))
using Printf

  Activating project at `~/M466`


In [3]:
Pkg.status()

Status `~/M466/Project.toml`
⌃ [336ed68f] CSV v0.10.16
  [a93c6f00] DataFrames v1.8.2
  [31c24e10] Distributions v0.25.131
  [91a5bcdd] Plots v1.41.7
⌃ [10745b16] Statistics v1.10.0
Info Packages marked with ⌃ have new versions available and may be upgradable.
Warning The project dependencies or compat requirements have changed since the manifest was last resolved. It is recommended to `Pkg.resolve()` or consider `Pkg.update()` if necessary.


In [5]:
function decode_bfloat16(bits; verbose=false)

    if verbose
        println("bfloat input:  ", bits)
    end

    # Remove spaces so either format works:
    # "0 10000000 1001001"
    # "0100000001001001"
    bits = replace(bits, " " => "")

    # Split the 16 bits into fields
    sign_bit      = bits[1]
    exponent_bits = bits[2:9]
    fraction_bits = bits[10:16]

    if verbose
        println("Sign bit:      ", sign_bit)
        println("Exponent bits: ", exponent_bits)
        println("Fraction bits: ", fraction_bits)
    end

    # ----- Sign -----
    s = sign_bit == '0' ? 1.0 : -1.0

    if verbose
        println("\nSign = ", s)
    end

    # ----- Exponent -----
    E = parse(Int, exponent_bits; base=2)
    e = E - 127

    if verbose
        println("Stored exponent E = ", E)
        println("Actual exponent e = ", E, " - 127 = ", e)
    end

    # ----- Fraction -----
    f = 0.0

    if verbose
        println("\nFraction calculation:")
    end

    for i in 1:length(fraction_bits)
        bit = fraction_bits[i]
        weight = 2.0^(-i)

        if verbose
            println("bit ", i,
                    " = ", bit,
                    ", weight = 2^(-", i, ") = ", weight)
        end

        if bit == '1'
            f += weight

            if verbose
                println("    add ", weight, " -> f = ", f)
            end
        end
    end

    # Normalized bfloat16 has an implicit leading 1
    significand = 1.0 + f

    if verbose
        println("\nFraction f = ", f)
        println("Significand 1 + f = ", significand)
    end

    # ----- Put everything together -----
    value = s * significand * 2.0^e

    if verbose
        println("\nFinal calculation:")
        println("x = (-1)^s × (1 + f) × 2^e")
        println("x = ", s, " × ", significand, " × 2^", e)
        println("x = ", value)
    end

    return value
end


decode_bfloat16 (generic function with 1 method)

In [6]:
function decimal_to_bfloat16(x::Float64; verbose=false)

    # Convert to Float32
    x32 = Float32(x)

    # Get the Float32 bit representation
    bits32 = reinterpret(UInt32, x32)
    bit_string = bitstring(bits32)

    # Split into retained and discarded bits
    kept_bits = bit_string[1:16]
    discarded_bits = bit_string[17:32]

    kept = parse(UInt16, kept_bits; base=2)
    discarded = parse(UInt16, discarded_bits; base=2)

    # Halfway point
    halfway = UInt16(0x8000)

    rounding_action = "round down"

    # Round to nearest
    if discarded > halfway
        kept += UInt16(1)
        rounding_action = "round up"
    end

    # Tie case: round to even
    if discarded == halfway
        if isodd(kept)
            kept += UInt16(1)
            rounding_action = "tie -> round up to even"
        else
            rounding_action = "tie -> round down to even"
        end
    end

    bits16 = kept
    b = bitstring(bits16)

    if verbose
        println("Input decimal:   ", x)
        @printf("Float32 value:   %.20f\n", x32)
        println("Float32 bits:    ", bit_string)
        println()

        println("Kept bits:       ", kept_bits)
        println("Discarded bits:  ", discarded_bits)
        println("Halfway:         ", bitstring(halfway))
        println("Rounding action: ", rounding_action)
        println()

        println("bfloat16 bits:   ", b)
        println("Sign:            ", b[1])
        println("Exponent:        ", b[2:9])
        println("Fraction:        ", b[10:16])
    end

    return b
end

decimal_to_bfloat16 (generic function with 1 method)

In [12]:
function bfloat_error(x::Float64; verbose=false)

    bits = decimal_to_bfloat16(x)
    xstar = decode_bfloat16(bits)

    epsilon = xstar - x

    if verbose
        println("bfloat16 bits: ", bits)
        println()

        println("Floating-point values:")
        @printf("x       = %.20f\n", x)
        @printf("x*      = %.20f\n", xstar)
        @printf("epsilon = %.20f\n", epsilon)
    end

    return xstar, epsilon
end

bfloat_error (generic function with 2 methods)

In [ ]:
#decimal_to_bfloat16(2.718; verbose=true)

## Question 1 Part (i)
Find the decimal equivalent of 0 10000000 1001001

In [8]:
decode_bfloat16("0 10000000 1001001")

3.140625

## Question1 Part (ii)
Find the decimal equivalent of 0 01111011 1001101

In [9]:
decode_bfloat16("0 01111011 1001101")

0.10009765625

## Question 1 Part(iii)
Compute the error ε = x∗ − x where x∗ is the bfloat16 closest to x = 2.718

In [15]:
#decimal_to_bfloat16(2.718)
bfloat_error(2.718, verbose = true)

bfloat16 bits: 0100000000101110

Floating-point values:
x       = 2.71799999999999997158
x*      = 2.71875000000000000000
epsilon = 0.00075000000000002842


(2.71875, 0.0007500000000000284)

## Question 1 Part (iv)
Compute the error ε = x∗ − x where x∗ is the bfloat16 closest to 12345.6

In [16]:
bfloat_error(12345.6, verbose = true)

bfloat16 bits: 0100011001000001

Floating-point values:
x       = 12345.60000000000036379788
x*      = 12352.00000000000000000000
epsilon = 6.39999999999963620212


(12352.0, 6.399999999999636)

This is an interesting result in that epsilon seems large (the absolute error).  This is due to the limit of 7 stored fraction bits for bfloat.  However the relative error is approx 0.05%  

In [21]:
function calc_cubes(x, y)

     if x == 0 && y == 0
        return 0.0
     end
    
    if abs(x) >= abs(y)
        return x * ((1 + (abs(y) / abs(x))^3)^(1/3))
    end

    if abs(y) >= abs(x)
        return y * ((1 + (abs(x) / abs(y))^3)^(1/3))
    end

end

calc_cubes (generic function with 1 method)

### Question 2: Use a more numerically stable calculation for the following quantities: 

#### Question 2 part (i)
x = 5.0 × 10e120 and y = 3.0 × 10e130

In [7]:
calc_cubes(5.0e120, 3.0e130)

3.0e130

In [17]:
## Retest with Bigfloat

setprecision(256) do
    x = BigFloat("5.0e120")
    y = BigFloat("3.0e130")

    r = (x / y)^3

    println("ratio cubed = ", r)
    println("1 + ratio cubed = ", 1 + r)
    println("result = ", y * cbrt(1 + r))
end

ratio cubed = 4.629629629629629629629629629629629629629629629629629629629629629629629629629515e-30
1 + ratio cubed = 1.000000000000000000000000000004629629629629629629629629629629629629629629629631
result = 3.000000000000000000000000000004629629629629629629629629629622485139460448102454e+130


#### Question 2 part (ii)
x = 1.0 and y = 3.0 × 10e130

In [11]:
calc_cubes(1.0, 3.0e130)

3.0e130

In [18]:
## Retest with Bigfloat

setprecision(256) do
    x = BigFloat("1.0")
    y = BigFloat("3.0e130")

    r = (x / y)^3

    println("ratio cubed = ", r)
    println("1 + ratio cubed = ", 1 + r)
    println("result = ", y * cbrt(1 + r))
end

ratio cubed = 3.703703703703703703703703703703703703703703703703703703703703703703703703703741e-392
1 + ratio cubed = 1.0
result = 3.000000000000000000000000000000000000000000000000000000000000000000000000000006e+130


#### Question 2 part (iii)
x = 5.0 × 10e120 and y = 1.0

In [22]:
calc_cubes(5.0e120, 1.0)

5.0e120

#### Question 2 part (iv)

In [23]:
calc_cubes(0.0, 0.0)

0.0

### Question 3: Perform forward and backward error analysis for: 
$$
f(x) = \sqrt[3]{x}
$$

## Forward and Backward Error Analysis for the Cube Root

Consider the function

$$
f(x)=\sqrt[3]{x}=x^{1/3}.
$$

Let the exact output be

$$
y=f(x).
$$

Suppose the input is perturbed from $x$ to $x^*$, where

$$
x^*=x(1+\delta).
$$

Then the relative error in the input is

$$
\boxed{
\frac{x^*-x}{x}=\delta
}
$$

This quantity is the relative backward error.
---

### Relative error on the output

The perturbed output is

$$
y^*=f(x^*).
$$

For the cube-root function,

$$
y^*
=
(x^*)^{1/3}.
$$

Substituting

$$
x^*=x(1+\delta),
$$

gives

$$
y^*
=
\left[x(1+\delta)\right]^{1/3}.
$$

Therefore,

$$
y^*
=
x^{1/3}(1+\delta)^{1/3}.
$$

Using the generalized binomial expansion,

$$
(1+\delta)^\alpha
=
1+\alpha\delta
+
\frac{\alpha(\alpha-1)}{2!}\delta^2
+\cdots.
$$

With $\alpha=1/3$,

$$
(1+\delta)^{1/3}
=
1+\frac{1}{3}\delta
-\frac{1}{9}\delta^2
+\cdots.
$$

For small $\delta$, neglect the second- and higher-order terms:

$$
(1+\delta)^{1/3}
\approx
1+\frac{1}{3}\delta.
$$

Thus,

$$
y^*
\approx
x^{1/3}
\left(
1+\frac{1}{3}\delta
\right).
$$

Since

$$
y=f(x)=x^{1/3},
$$

we obtain

$$
y^*-y
\approx
\frac{1}{3}x^{1/3}\delta.
$$

Dividing by $y=x^{1/3}$ gives the relative output error:

$$
\boxed{
\frac{y^*-y}{y}
\approx
\frac{1}{3}\delta
}
$$

The quantity

$$
\boxed{
\frac{y^*-y}{y}
}
$$

is the **relative forward error**.

Therefore,

$$
\boxed{
\text{relative forward error}
\approx
\frac{1}{3}
\text{ relative backward error}.
}
$$

---

## Relative Condition Number

The relative condition number measures the ratio of relative output error to relative input error as the input perturbation approaches zero:

$$
\kappa_f(x)
=
\lim_{\Delta x\to 0}
\left|
\frac{
\dfrac{f(x+\Delta x)-f(x)}{f(x)}
}{
\dfrac{\Delta x}{x}
}
\right|.
$$

Rearranging,

$$
\kappa_f(x)
=
\left|
\frac{x}{f(x)}
\right|
\lim_{\Delta x\to0}
\left|
\frac{f(x+\Delta x)-f(x)}{\Delta x}
\right|.
$$

The limit is the definition of the derivative:

$$
f'(x)
=
\lim_{\Delta x\to0}
\frac{f(x+\Delta x)-f(x)}{\Delta x}.
$$

Therefore,

$$
\boxed{
\kappa_f(x)
=
\left|
\frac{x f'(x)}{f(x)}
\right|.
}
$$

For

$$
f(x)=x^{1/3},
$$

we have

$$
f'(x)
=
\frac{1}{3}x^{-2/3}.
$$

Hence,

$$
\kappa_f(x)
=
\left|
\frac{
x\left(\frac{1}{3}x^{-2/3}\right)
}{
x^{1/3}
}
\right|.
$$

Simplifying,

$$
\boxed{
\kappa_f(x)=\frac{1}{3}.
}
$$

---

## Relationship Between Forward and Backward Error

To first order, the relative condition number connects the relative backward error and relative forward error:

$$
\boxed{
\left|
\frac{y^*-y}{y}
\right|
\approx
\kappa_f(x)
\left|
\frac{x^*-x}{x}
\right|.
}
$$

Equivalently,

$$
\boxed{
\text{relative forward error}
\approx
\kappa_f(x)
\times
\text{relative backward error}.
}
$$

For the cube-root function,

$$
\kappa_f(x)=\frac{1}{3},
$$

so

$$
\boxed{
\text{relative forward error}
\approx
\frac{1}{3}
\text{ relative backward error}.
}
$$

Solving for the backward error gives

$$
\boxed{
\text{relative backward error}
\approx
3
\text{ relative forward error}.
}
$$

Thus, the cube-root function **damps relative error by approximately a factor of three**.

### Summary

$$
\boxed{
\underbrace{
\left|\frac{x^*-x}{x}\right|
}_{\text{relative backward error}}
\quad
\xrightarrow{\;\;f\;\;}
\quad
\underbrace{
\left|\frac{y^*-y}{y}\right|
}_{\text{relative forward error}}
}
$$

with

$$
\boxed{
\text{Forward Error}
\approx
\kappa_f(x)
\times
\text{Backward Error}.
}
$$

For $f(x)=x^{1/3}$,

$$
\boxed{
\kappa_f(x)=\frac{1}{3}.
}
$$


# Assignment 1 Notes: 

## Converting a Decimal Number to bfloat16

A decimal number such as

$$
x = 2.718
$$

cannot, in general, be represented exactly as a binary floating-point number. This is
similar to the fact that $1/3$ cannot be represented exactly with a finite number of
decimal digits.

The conversion to bfloat16 can be viewed as a sequence of approximations:

$$
\text{Decimal} \rightarrow \text{Float64} \rightarrow \text{Float32}
\rightarrow \text{bfloat16}.
$$

### Step 1: Decimal to Float64

When `2.718` is entered in Julia, Julia represents the number as a `Float64`.

Although Julia normally displays this value simply as

```text
2.718
```

the actual stored value can be displayed with additional precision using

```julia
@printf("%.20f\n", 2.718)
```

which gives approximately

```text
2.71799999999999997158
```

Thus, even the original `Float64` is already a binary approximation to the
decimal number $2.718$.

This occurs because numbers stored in binary floating-point form are constructed
from powers of 2. A decimal fraction such as $2.718$ does not have a finite binary
representation, so it must be rounded to the nearest representable `Float64` value.

---

### Step 2: Float64 to Float32

Our bfloat16 conversion first converts the number to `Float32`:

```julia
x32 = Float32(x)
```

A `Float32` contains 32 bits divided into

$$
1 \text{ sign bit} + 8 \text{ exponent bits} + 23 \text{ fraction bits}.
$$

Because `Float32` has fewer fraction bits than `Float64`, another rounding occurs.

For $x=2.718$, the stored `Float32` value is

```text
2.71799993515014648438
```

and its 32-bit representation is

```text
01000000001011011111001110110110
```

Separating this into the IEEE 754 fields gives

```text
0 | 10000000 | 01011011111001110110110
```

where

```text
sign     = 0
exponent = 10000000
fraction = 01011011111001110110110
```

The sign bit is 0, so the number is positive.

The stored exponent is

$$
E = (10000000)_2 = 128.
$$

Using the Float32 exponent bias of 127,

$$
e = E - 127 = 128 - 127 = 1.
$$

---

### Step 3: Float32 to bfloat16

A bfloat16 number contains only 16 bits:

$$
1 \text{ sign bit} + 8 \text{ exponent bits} + 7 \text{ fraction bits}.
$$

An important feature of bfloat16 is that it uses the same 8-bit exponent field
and exponent bias as `Float32`. The major reduction in size comes from reducing
the fraction from 23 bits to 7 bits.

Therefore, when converting a `Float32` to bfloat16, the upper 16 bits contain
exactly the fields needed for bfloat16:

```text
Float32:

0 | 10000000 | 0101101 1111001110110110
                 ^             ^
              retained      discarded
```

The first 16 bits are retained:

```text
0100000000101101
```

and the remaining 16 fraction bits are discarded:

```text
1111001110110110
```

Simply discarding these bits would truncate the number. Instead, we want the
**nearest representable bfloat16 value**, so the discarded bits are used to
determine whether the retained bits should be rounded up.

---

### Step 4: Determine the Rounding Direction

With 16 discarded bits, the halfway point is

```text
1000000000000000
```

For $x=2.718$, the discarded bits are

```text
1111001110110110
```

Comparing them gives

```text
1111001110110110
>
1000000000000000
```

so the discarded portion is greater than halfway.

Therefore, the nearest bfloat16 value is obtained by **rounding up**.

Before rounding, the retained bits are

```text
0 | 10000000 | 0101101
```

After rounding, they become

```text
0 | 10000000 | 0101110
```

Thus the final bfloat16 bit pattern is

```text
0100000000101110
```

---

### Step 5: Decode the Resulting bfloat16

The bfloat16 representation

```text
0 | 10000000 | 0101110
```

has

```text
sign     = 0
exponent = 10000000
fraction = 0101110
```

The exponent is

$$
e = 128 - 127 = 1.
$$

The fraction is

$$
f =
0(2^{-1})
+1(2^{-2})
+0(2^{-3})
+1(2^{-4})
+1(2^{-5})
+1(2^{-6})
+0(2^{-7}).
$$

Therefore,

$$
f =
\frac{1}{4}
+\frac{1}{16}
+\frac{1}{32}
+\frac{1}{64}
=0.359375.
$$

For a normalized bfloat16 number, there is an implicit leading 1, so the
significand is

$$
1+f=1.359375.
$$

The represented value is therefore

$$
x^*
=
(+1)(1.359375)2^1
=
2.71875.
$$

Thus,

$$
\boxed{x^*=2.71875}.
$$

---

## Summary of the Conversion

The complete conversion can be viewed as

$$
2.718
\longrightarrow
2.71799999999999997158\ldots
\qquad \text{(Float64)}
$$

then

$$
2.71799999999999997158\ldots
\longrightarrow
2.71799993515014648438\ldots
\qquad \text{(Float32)}
$$

and finally

$$
2.71799993515014648438\ldots
\longrightarrow
2.71875
\qquad \text{(bfloat16)}.
$$

Each change occurs because a floating-point format contains only a finite number
of fraction bits. Reducing the number of available fraction bits reduces
precision and requires rounding to a nearby representable number.

For this example, the final bfloat16 representation is

```text
0 | 10000000 | 0101110
```

or, as a 16-bit string,

```text
0100000000101110
```

representing

$$
\boxed{2.71875}.
$$

The next quantity of interest is the floating-point representation error,

$$
\epsilon = x^* - x,
$$

which measures the difference between the bfloat16 approximation $x^*$ and the
original value $x$.